# Data Preparation & EDA
## What this notebook does

It produces:
- `data/train_indices.npy` - training row indices
- `data/val_indices.npy` - validation row indices  
- `data/test_indices.npy` - test row indices
- `data/train_augmented.csv` - CDA-augmented training set
- `data/class_weights.json` - class weights computed from augmented training set
- `data/data_summary.json` - EDA stats for thesis documentation
- `data/eda_plots/` - visualisation plots



## 1. Install packages

In [ ]:
%pip install pandas numpy scikit-learn matplotlib seaborn --quiet

## 2. Imports

In [ ]:
import os
import re
import json
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from sklearn.model_selection import train_test_split

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.dpi"] = 120

## 3. Configuration

In [ ]:
# Paths
BASE_DIR    = r"C:\Users\scanc\Desktop\v3"
DATA_PATH   = os.path.join(BASE_DIR, "jigsawdataset", "train.csv")
OUT_DIR     = os.path.join(BASE_DIR, "data")
PLOTS_DIR   = os.path.join(OUT_DIR, "eda_plots")

os.makedirs(OUT_DIR,   exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)

# Experiment settings
SEED            = 42
SAMPLE_SIZE     = 100_000   # None = full dataset
TOXICITY_THRESH = 0.5       # binarisation threshold
IDENTITY_THRESH = 0.5       # identity mention threshold

random.seed(SEED)
np.random.seed(SEED)

print(f"Base dir    : {BASE_DIR}")
print(f"Data path   : {DATA_PATH}")
print(f"Output dir  : {OUT_DIR}")
print(f"Sample size : {SAMPLE_SIZE or 'full dataset'}")
print(f"Seed        : {SEED}")

## 4. Load Jigsaw dataset

In [ ]:
print("Loading dataset...")
data = pd.read_csv(DATA_PATH)
print(f"Full dataset shape : {data.shape}")
print(f"Columns            : {len(data.columns)}")
print()

# Identity annotation columns
IDENTITY_COLUMNS = [
    "male", "female", "homosexual_gay_or_lesbian",
    "christian", "jewish", "muslim",
    "black", "white", "psychiatric_or_mental_illness",
]

existing_id_cols = [c for c in IDENTITY_COLUMNS if c in data.columns]
print(f"Identity columns found: {len(existing_id_cols)} / {len(IDENTITY_COLUMNS)}")
print(f"Missing             : {[c for c in IDENTITY_COLUMNS if c not in data.columns]}")
print()
print("Sample columns:")
print(data.columns.tolist()[:10])

## 5. EDA - Class distribution

Binarise the continuous toxicity score at 0.5 and examine the class balance.

In [ ]:
# Binarise
data["label"] = (data["target"] >= TOXICITY_THRESH).astype(int)

counts = data["label"].value_counts().sort_index()
pcts   = data["label"].value_counts(normalize=True).sort_index() * 100

print("=== CLASS DISTRIBUTION (full dataset) ===")
print(f"  Non-toxic (0) : {counts[0]:>10,}  ({pcts[0]:.2f}%)")
print(f"  Toxic     (1) : {counts[1]:>10,}  ({pcts[1]:.2f}%)")
print(f"  Total         : {len(data):>10,}")
print(f"  Imbalance ratio (non-toxic:toxic) = {counts[0]/counts[1]:.1f}:1")

# Plot
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(["Non-toxic", "Toxic"], counts.values,
              color=["#4575b4", "#d73027"], edgecolor="white", width=0.5)
for bar, pct in zip(bars, pcts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5000,
            f"{pct:.1f}%", ha="center", fontsize=11)
ax.set_title("Class Distribution - Full Jigsaw Dataset", fontweight="bold")
ax.set_ylabel("Number of comments")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "class_distribution.png"), bbox_inches="tight")
plt.show()

## 6. EDA - Identity annotation coverage

In [ ]:
# How many rows have any identity annotation?
has_any_identity = data[existing_id_cols].notna().any(axis=1)
identity_coverage = has_any_identity.mean() * 100

print(f"=== IDENTITY ANNOTATION COVERAGE ===")
print(f"  Rows with any identity annotation : {has_any_identity.sum():,} ({identity_coverage:.2f}%)")
print(f"  Rows without identity annotation  : {(~has_any_identity).sum():,} ({100-identity_coverage:.2f}%)")
print()

# Per-group coverage
print("Per-group annotation coverage:")
for col in existing_id_cols:
    non_null = data[col].notna().sum()
    mentions = (data[col] >= IDENTITY_THRESH).sum()
    pct_non_null = 100 * non_null / len(data)
    pct_mention  = 100 * mentions / len(data)
    print(f"  {col:<40} annotated: {non_null:>8,} ({pct_non_null:.1f}%)   "
          f"mentions(>={IDENTITY_THRESH}): {mentions:>6,} ({pct_mention:.2f}%)")

# Plot
mention_counts = [(data[c] >= IDENTITY_THRESH).sum() for c in existing_id_cols]
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(existing_id_cols, mention_counts, color="#4575b4", edgecolor="white")
ax.set_xlabel("Number of comments with clear identity mention")
ax.set_title("Identity Group Mention Counts (threshold = 0.5)", fontweight="bold")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "identity_coverage.png"), bbox_inches="tight")
plt.show()

## 7. EDA - Toxicity rate: identity mentions vs non-mentions

This is the shortcut-learning risk documented in Section 3.4 of the methodology.

In [ ]:
identity_subset = data[has_any_identity]
non_identity_subset = data[~has_any_identity]

tox_with    = identity_subset["label"].mean() * 100
tox_without = non_identity_subset["label"].mean() * 100
ratio       = tox_with / tox_without

print("=== TOXICITY RATE BY IDENTITY MENTION ===")
print(f"  With identity mention    : {tox_with:.2f}%")
print(f"  Without identity mention : {tox_without:.2f}%")
print(f"  Ratio                    : {ratio:.1f}x higher with identity mention")
print()
print("This confirms the shortcut-learning risk:")
print("   A classifier can learn to associate identity terms with toxicity.")

# Per-group toxicity rates
print()
print("Per-group toxicity rates:")
group_stats = []
for col in existing_id_cols:
    group = data[data[col] >= IDENTITY_THRESH]
    if len(group) >= 50:
        tox_rate = group["label"].mean() * 100
        group_stats.append((col, len(group), tox_rate))
        print(f"  {col:<40} n={len(group):>6,}  toxic rate={tox_rate:.1f}%")

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
names  = [g[0] for g in group_stats]
rates  = [g[2] for g in group_stats]
colors = ["#d73027" if r > tox_without else "#4575b4" for r in rates]
ax.barh(names, rates, color=colors, edgecolor="white")
ax.axvline(tox_without, color="black", linestyle="--", linewidth=1.5,
           label=f"Non-identity baseline ({tox_without:.1f}%)")
ax.set_xlabel("Toxicity rate (%)")
ax.set_title("Toxicity Rate per Identity Group vs Non-identity Baseline",
             fontweight="bold")
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "toxicity_rate_by_group.png"), bbox_inches="tight")
plt.show()

## 8. EDA - Comment length distribution

In [ ]:
# Sample for speed
sample_for_len = data.sample(min(50000, len(data)), random_state=SEED)
word_lengths   = sample_for_len["comment_text"].dropna().str.split().str.len()

print("=== COMMENT LENGTH (words) ===")
print(f"  Mean   : {word_lengths.mean():.0f}")
print(f"  Median : {word_lengths.median():.0f}")
print(f"  p95    : {word_lengths.quantile(0.95):.0f}")
print(f"  p99    : {word_lengths.quantile(0.99):.0f}")
print(f"  Max    : {word_lengths.max():.0f}")
print()
print(f"  Comments under 128 words  : {(word_lengths <= 128).mean()*100:.1f}%")
print(f"  Comments under 64 words   : {(word_lengths <= 64).mean()*100:.1f}%")
print("  -> MAX_LENGTH=128 covers most comments")

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(word_lengths.clip(upper=300), bins=50, color="#4575b4", edgecolor="white")
ax.axvline(128, color="#d73027", linestyle="--", linewidth=1.5, label="MAX_LENGTH=128")
ax.set_xlabel("Comment length (words)")
ax.set_ylabel("Count")
ax.set_title("Comment Length Distribution (capped at 300)", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "comment_lengths.png"), bbox_inches="tight")
plt.show()

## 9. Prepare dataset - keep needed columns

In [ ]:
needed = ["id", "comment_text", "label"] + existing_id_cols
data_clean = data[needed].dropna(subset=["comment_text", "label"]).reset_index(drop=True)
print(f"Cleaned dataset shape: {data_clean.shape}")
print(f"Columns: {data_clean.columns.tolist()}")

## 10. Sample dataset

In [ ]:
if SAMPLE_SIZE is not None and SAMPLE_SIZE < len(data_clean):
    data_sample = data_clean.sample(n=SAMPLE_SIZE, random_state=SEED).reset_index(drop=True)
    print(f"Sampled {SAMPLE_SIZE:,} rows from {len(data_clean):,}")
else:
    data_sample = data_clean.copy().reset_index(drop=True)
    print(f"Using full dataset: {len(data_sample):,} rows")

print(f"\nSample label distribution:")
print(data_sample["label"].value_counts())
print(data_sample["label"].value_counts(normalize=True).round(4))

## 11. Create shared train / validation / test split

This is the only place the split is created. All training notebooks load these indices.

In [ ]:
# Step 1: split off test (20%)
train_val_idx, test_idx = train_test_split(
    data_sample.index.tolist(),
    test_size=0.20,
    random_state=SEED,
    stratify=data_sample["label"],
)

# Step 2: split remaining into train (70%) and val (10%)
train_idx, val_idx = train_test_split(
    train_val_idx,
    test_size=0.125,   # 10% of total = 12.5% of 80%
    random_state=SEED,
    stratify=data_sample.loc[train_val_idx, "label"],
)

train_df = data_sample.loc[train_idx].reset_index(drop=True)
val_df   = data_sample.loc[val_idx].reset_index(drop=True)
test_df  = data_sample.loc[test_idx].reset_index(drop=True)

print(f"Train : {len(train_df):,}  ({len(train_df)/len(data_sample)*100:.1f}%)")
print(f"Val   : {len(val_df):,}   ({len(val_df)/len(data_sample)*100:.1f}%)")
print(f"Test  : {len(test_df):,}  ({len(test_df)/len(data_sample)*100:.1f}%)")

print(f"\nTrain label %: {train_df['label'].value_counts(normalize=True).round(3).to_dict()}")
print(f"Val   label %: {val_df['label'].value_counts(normalize=True).round(3).to_dict()}")
print(f"Test  label %: {test_df['label'].value_counts(normalize=True).round(3).to_dict()}")

# Save indices
np.save(os.path.join(OUT_DIR, "train_indices.npy"), np.array(train_idx))
np.save(os.path.join(OUT_DIR, "val_indices.npy"),   np.array(val_idx))
np.save(os.path.join(OUT_DIR, "test_indices.npy"),  np.array(test_idx))

# Save raw splits (without CDA) for loading in notebooks
train_df.to_csv(os.path.join(OUT_DIR, "train_raw.csv"),      index=False)
val_df.to_csv(  os.path.join(OUT_DIR, "val.csv"),             index=False)
test_df.to_csv( os.path.join(OUT_DIR, "test.csv"),            index=False)
data_sample.to_csv(os.path.join(OUT_DIR, "data_sample.csv"), index=False)

print("\nSaved: train_indices.npy, val_indices.npy, test_indices.npy")
print("Saved: train_raw.csv, val.csv, test.csv, data_sample.csv")

## 12. CDA - Explicit swap dictionaries

Each swap is intentionally defined. No `zip()` - every pair has been manually verified to be semantically appropriate.

In [ ]:
# Gender swaps (fully bidirectional)
GENDER_SWAPS = {
    "he": "she",       "she": "he",
    "him": "her",      "her": "him",
    "his": "hers",     "hers": "his",
    "man": "woman",    "woman": "man",
    "men": "women",    "women": "men",
    "boy": "girl",     "girl": "boy",
    "boys": "girls",   "girls": "boys",
    "male": "female",  "female": "male",
    "males": "females","females": "males",
    "father": "mother","mother": "father",
    "son": "daughter", "daughter": "son",
    "brother": "sister","sister": "brother",
    "husband": "wife", "wife": "husband",
}

# Religion swaps - only clear identity labels, not associated concepts
RELIGION_SWAPS = {
    "muslim": "christian",    "christian": "muslim",
    "muslims": "christians",  "christians": "muslims",
    "islamic": "christian",   "islam": "christianity",
    "christianity": "islam",
    "mosque": "church",       "church": "mosque",
    "jewish": "christian",    "jew": "christian",
    "jews": "christians",     "judaism": "christianity",
    "synagogue": "church",
}

# Race swaps - only clear identity labels
RACE_SWAPS = {
    "black": "white",          "white": "black",
    "blacks": "whites",        "whites": "blacks",
    "african american": "white person",
    "african-american": "white",
}

# Combine all swap dictionaries
ALL_SWAPS = {}
ALL_SWAPS.update(GENDER_SWAPS)
ALL_SWAPS.update(RELIGION_SWAPS)
ALL_SWAPS.update(RACE_SWAPS)

print(f"Gender swaps   : {len(GENDER_SWAPS)} pairs")
print(f"Religion swaps : {len(RELIGION_SWAPS)} pairs")
print(f"Race swaps     : {len(RACE_SWAPS)} pairs")
print(f"Total swaps    : {len(ALL_SWAPS)} pairs")
print()
print("Sample swaps:")
for k, v in list(ALL_SWAPS.items())[:8]:
    print(f"  '{k}' -> '{v}'"  )

## 13. CDA augmentation function

For each training comment that contains a swappable term, generate a counterfactual copy with all matching terms replaced. Both copies keep the same toxicity label.

In [ ]:
def apply_swaps(text, swap_dict):
    """
    Apply all swap_dict substitutions to text using word-boundary regex.
    Longer terms are matched first to avoid partial replacements.
    Returns the swapped text, or None if no swap was made.
    """
    result    = text
    swapped   = False
    # Sort by length descending so multi-word terms match before substrings
    sorted_pairs = sorted(swap_dict.items(), key=lambda x: -len(x[0]))

    for src, tgt in sorted_pairs:
        pattern = r'\b' + re.escape(src) + r'\b'
        new_result, n = re.subn(pattern, tgt, result, flags=re.IGNORECASE)
        if n > 0:
            result  = new_result
            swapped = True

    return result if swapped else None


def generate_cda(df, swap_dict, text_col="comment_text",
                  label_col="label", seed=42):
    """
    Generate CDA dataset.
    For each row where at least one swap applies, produce a counterfactual copy.
    Returns original + augmented rows shuffled together, with pair_id tracking.
    """
    augmented = []
    pair_id   = 0

    original_rows = []
    for _, row in df.iterrows():
        text        = str(row[text_col])
        swapped     = apply_swaps(text, swap_dict)

        if swapped is not None and swapped != text:
            # Original gets a pair_id
            orig           = row.copy()
            orig["pair_id"] = pair_id
            original_rows.append(orig)

            # Counterfactual gets the same pair_id
            aug             = row.copy()
            aug[text_col]   = swapped
            aug["pair_id"]  = pair_id
            augmented.append(aug)
            pair_id        += 1
        else:
            orig            = row.copy()
            orig["pair_id"] = -1   # no counterfactual
            original_rows.append(orig)

    orig_df = pd.DataFrame(original_rows)
    aug_df  = pd.DataFrame(augmented) if augmented else pd.DataFrame()

    if not aug_df.empty:
        combined = pd.concat([orig_df, aug_df], ignore_index=True)
    else:
        combined = orig_df.copy()

    combined = combined.sample(frac=1, random_state=seed).reset_index(drop=True)

    print(f"  Original rows     : {len(orig_df):,}")
    print(f"  Augmented rows    : {len(aug_df):,}")
    print(f"  Total rows        : {len(combined):,}")
    print(f"  Pairs created     : {pair_id:,}")
    print(f"  Augmentation rate : {len(aug_df)/len(orig_df)*100:.1f}%")
    return combined



## 14. Apply CDA to training set only

Validation and test sets are never augmented.

In [ ]:
print("Applying CDA to training set...")
train_augmented = generate_cda(
    train_df, ALL_SWAPS,
    text_col="comment_text", label_col="label", seed=SEED
)

print(f"\nAugmented label distribution:")
print(train_augmented["label"].value_counts())
print(train_augmented["label"].value_counts(normalize=True).round(3))

# Save augmented training set
aug_path = os.path.join(OUT_DIR, "train_augmented.csv")
train_augmented.to_csv(aug_path, index=False)
print(f"\nSaved: {aug_path}")

## 15. Compute class weights

Computed once from the augmented training set. All training notebooks load these values.
Using inverse-frequency weighting: `weight_c = total / (n_classes × count_c)`

In [ ]:
label_counts = train_augmented["label"].value_counts().sort_index()
n_classes    = len(label_counts)
total        = label_counts.sum()

class_weights = {}
for label, count in label_counts.items():
    class_weights[int(label)] = round(float(total / (n_classes * count)), 6)

print("=== CLASS WEIGHTS (from augmented training set) ===")
for label, weight in class_weights.items():
    name = "non-toxic" if label == 0 else "toxic"
    print(f"  Class {label} ({name}) : count={label_counts[label]:,}  weight={weight:.4f}")

weights_path = os.path.join(OUT_DIR, "class_weights.json")
with open(weights_path, "w") as f:
    json.dump(class_weights, f, indent=2)
print(f"\nSaved: {weights_path}")

## 16. Save data summary for thesis documentation

In [ ]:
has_id     = data_sample[existing_id_cols].notna().any(axis=1)
tox_with   = data_sample[has_id]["label"].mean() * 100
tox_without= data_sample[~has_id]["label"].mean() * 100

summary = {
    "dataset": {
        "total_rows_full"          : len(data),
        "sample_size"              : len(data_sample),
        "seed"                     : SEED,
        "toxicity_threshold"       : TOXICITY_THRESH,
        "identity_threshold"       : IDENTITY_THRESH,
    },
    "class_distribution": {
        "non_toxic_count"          : int(data_sample["label"].value_counts()[0]),
        "toxic_count"              : int(data_sample["label"].value_counts()[1]),
        "non_toxic_pct"            : round(float(data_sample["label"].value_counts(normalize=True)[0]*100), 2),
        "toxic_pct"                : round(float(data_sample["label"].value_counts(normalize=True)[1]*100), 2),
        "imbalance_ratio"          : round(float(data_sample["label"].value_counts()[0] /
                                           data_sample["label"].value_counts()[1]), 2),
    },
    "identity_coverage": {
        "rows_with_any_identity"   : int(has_id.sum()),
        "identity_coverage_pct"    : round(float(has_id.mean()*100), 2),
        "toxicity_rate_with_identity"    : round(tox_with, 2),
        "toxicity_rate_without_identity" : round(tox_without, 2),
        "toxicity_rate_ratio"            : round(tox_with / tox_without, 2),
    },
    "splits": {
        "train_size"               : len(train_df),
        "val_size"                 : len(val_df),
        "test_size"                : len(test_df),
        "train_pct"                : round(len(train_df)/len(data_sample)*100, 1),
        "val_pct"                  : round(len(val_df)/len(data_sample)*100, 1),
        "test_pct"                 : round(len(test_df)/len(data_sample)*100, 1),
    },
    "cda": {
        "original_train_rows"      : len(train_df),
        "augmented_train_rows"     : len(train_augmented),
        "augmented_rows_added"     : len(train_augmented) - len(train_df),
        "augmentation_rate_pct"    : round((len(train_augmented)-len(train_df))/len(train_df)*100, 1),
        "total_swap_pairs"         : len(ALL_SWAPS),
        "gender_swaps"             : len(GENDER_SWAPS),
        "religion_swaps"           : len(RELIGION_SWAPS),
        "race_swaps"               : len(RACE_SWAPS),
    },
    "class_weights"                : class_weights,
    "identity_columns"             : existing_id_cols,
}

summary_path = os.path.join(OUT_DIR, "data_summary.json")
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)

print("=== DATA SUMMARY ===")
print(json.dumps(summary, indent=2))
print(f"\nSaved: {summary_path}")

## 17. Output checklist

Verify all files were created before running any training notebook.

In [ ]:
expected_files = [
    "train_indices.npy",
    "val_indices.npy",
    "test_indices.npy",
    "train_raw.csv",
    "train_augmented.csv",
    "val.csv",
    "test.csv",
    "data_sample.csv",
    "class_weights.json",
    "data_summary.json",
]

print("=== OUTPUT FILE CHECKLIST ===")
all_ok = True
for fname in expected_files:
    fpath  = os.path.join(OUT_DIR, fname)
    exists = os.path.exists(fpath)
    size   = os.path.getsize(fpath) / 1024 if exists else 0
    icon   = "v" if exists else "X"
    if not exists: all_ok = False
    print(f"  {icon}  {fname:<35} {size:>8.1f} KB")

print()
if all_ok:
    print("All files present. Ready to run training notebooks.")
else:
    print("Some files missing - re-run the notebook.")
print()
print("=== NEXT STEPS ===")
print("  Load in every training notebook:")
print(f"    DATA_DIR = r'{OUT_DIR}'")
print("    train_aug = pd.read_csv(os.path.join(DATA_DIR, 'train_augmented.csv'))")
print("    val_df    = pd.read_csv(os.path.join(DATA_DIR, 'val.csv'))")
print("    test_df   = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))")
print("    with open(os.path.join(DATA_DIR, 'class_weights.json')) as f:")
print("        class_weights = json.load(f)")